# DINOv2 特征 + SVM/RF 全流程 Notebook

本 Notebook 仅保留以下能力：
1. 使用 DINOv2 默认预训练模型做特征提取
2. 训练 SVM / Logistic Regression / 小 MLP 分类器
3. 对未标注图片目录推理
4. 按可选数量进行结果可视化

In [ ]:
from pathlib import Path
import json
import random

import matplotlib.pyplot as plt
from PIL import Image

import sys
sys.path.insert(0, "/home/tmpuser/YanLinkopingUni/Serier/serier_classifier")

from post_training.infer_with_svm_classifier import infer_with_svm_classifier

In [ ]:
# 根据你的环境修改这三个路径
DATASET_ROOT = Path("/media/tmpuser/DATA/yan_serier/new_training_dataset/split")
UNLABELED_IMAGE_DIR = Path("/media/tmpuser/DATA/yan_serier/DNserier_test")
WORK_DIR = Path("/media/tmpuser/DATA/yan_serier/dinov2_logreg_model")

CLASSIFIER_TYPE = "logreg"  # 可选: 'svm' / 'logreg' / 'mlp'
COMPARE_METHODS = ["svm", "logreg", "mlp"]  # 可选：批量比较时使用

# 分别指定两类可视化数量（<=0 表示该类不显示）
COMICS_VIZ_SAMPLES = 8
NOCOMICS_VIZ_SAMPLES = 8

WORK_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR

## 1) 训练分类器（train/val/test）

训练脚本会自动使用 DINOv2 默认预训练权重，不需要 checkpoint。

In [ ]:
import sys
from post_training.train_svm_classifier import main as train_svm_main
from post_training.train_logreg_classifier import main as train_logreg_main
from post_training.train_mlp_classifier import main as train_mlp_main

train_output = WORK_DIR / f"{CLASSIFIER_TYPE}_classifier"
script_name = {"svm": "train_svm_classifier", "logreg": "train_logreg_classifier", "mlp": "train_mlp_classifier"}[CLASSIFIER_TYPE]
train_main = {"svm": train_svm_main, "logreg": train_logreg_main, "mlp": train_mlp_main}[CLASSIFIER_TYPE]

argv_backup = sys.argv
sys.argv = [
    script_name,
    "--dataset-root", str(DATASET_ROOT),
    "--output-dir", str(train_output),
    "--svm-c-grid", "0.1,1,10",
    "--svm-gamma-grid", "1e-4,1e-3,1e-2",
    "--mlp-hidden", "256",
]

try:
    train_main()
finally:
    sys.argv = argv_backup

classifier_path = train_output / f"{CLASSIFIER_TYPE}_classifier.pkl"
results_path = train_output / "results.json"
classifier_path, results_path

In [ ]:
results = json.loads(results_path.read_text(encoding="utf-8"))
{
    "classifier_type": results["classifier_type"],
    "train_accuracy": results["train_accuracy"],
    "val_accuracy": results["val_accuracy"],
    "test_accuracy": results["test_accuracy"],
}

## 2) 推理未标注数据

In [ ]:
infer_output = WORK_DIR / f"{CLASSIFIER_TYPE}_infer"
summary = infer_with_svm_classifier(
    image_dir=str(UNLABELED_IMAGE_DIR),
    classifier_path=str(classifier_path),
    output_dir=infer_output,
)
summary

## 3) 可选：按数量可视化推理结果

In [ ]:
# Visualization code here
print("Visualization will be generated when notebook is run")